# Part 1: Clustering
### Farthest First (k-center) and k-Means++ Algorithms
**Dataset:** UCI Spam Dataset (4601 points, 58 dimensions)

In [ ]:
# Install PySpark (run once in Colab)
!pip install pyspark --quiet

In [ ]:
import time
import random
import numpy as np
from pyspark import SparkContext, SparkConf

# Initialize SparkContext (use getOrCreate to avoid errors on re-run)
conf = SparkConf().setAppName("Clustering").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf)
sc.setLogLevel("ERROR")  # suppress verbose Spark logs

print("Spark version:", sc.version)
print("SparkContext initialized successfully.")

Spark version: 4.0.2
SparkContext initialized successfully.


In [ ]:
# Download the UCI Spambase dataset from UCI ML Repository
import urllib.request
import os

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data"
filename = "spambase.data"

if not os.path.exists(filename):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, filename)
    print(f"Downloaded to {filename}")
else:
    print(f"Dataset already exists: {filename}")

# Quick peek at the file
with open(filename) as f:
    first_line = f.readline()
print("First line (truncated):", first_line[:120])
print("Number of features in first line:", len(first_line.strip().split(',')))

Downloaded to spambase.data
First line (truncated): 0,0.64,0.64,0,0.32,0,0,0,0,0,0,0.64,0,0,0,0.32,0,1.29,1.93,0,0.96,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,
Number of features in first line: 58


In [ ]:
def readVectorsSeq(filename):

    vectors = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # Parse all values; drop the last column (class label)
            vals = list(map(float, line.split(',')))
            vectors.append(np.array(vals[:-1]))  # 57 features
    return vectors


# Load the dataset
P = readVectorsSeq(filename)
print(f"Total points loaded : {len(P)}")
print(f"Dimensions per point: {len(P[0])}")
print(f"Sample vector (first 5 dims): {P[0][:5]}")

Total points loaded : 4601
Dimensions per point: 57
Sample vector (first 5 dims): [0.   0.64 0.64 0.   0.32]


In [ ]:
def sqdist(a, b):

    diff = a - b
    return float(np.dot(diff, diff))


# Quick sanity check
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 6.0, 3.0])
print("sqdist([1,2,3], [4,6,3]) =", sqdist(a, b))  # Expected: 9+16+0 = 25

sqdist([1,2,3], [4,6,3]) = 25.0


In [ ]:
def kcenter(P, k):

    n = len(P)
    assert k <= n, "k cannot exceed the number of points"

    # Step 1: Initialise with the first point as the first center
    centers = [P[0]]

    # Step 2: Initialise min_dist array — distance from each point to nearest center
    # After choosing P[0], distance from every point to the first center
    min_dist = np.array([sqdist(p, P[0]) for p in P], dtype=float)

    # Step 3: Iteratively pick the farthest point
    for _ in range(k - 1):
        # The next center is the point with max distance to any current center
        next_idx = int(np.argmax(min_dist))
        next_center = P[next_idx]
        centers.append(next_center)

        # Update min_dist: compare with distance to the newly added center — O(|P|)
        for i in range(n):
            d = sqdist(P[i], next_center)
            if d < min_dist[i]:
                min_dist[i] = d

    return centers


print("kcenter function defined.")

kcenter function defined.


In [ ]:
def kmeansPP(P, k):

    n = len(P)
    assert k <= n, "k cannot exceed the number of points"

    # Step 1: Choose first center uniformly at random
    first_idx = random.randint(0, n - 1)
    centers = [P[first_idx]]

    # Step 2: Initialise D² distances
    D = np.array([sqdist(p, P[first_idx]) for p in P], dtype=float)

    # Step 3: Iteratively sample centers with probability proportional to D²
    for _ in range(k - 1):
        total = D.sum()
        if total == 0:
            # All remaining points coincide with chosen centers — pick randomly
            next_idx = random.randint(0, n - 1)
        else:
            # Normalise to get a probability distribution and sample
            probs = D / total
            next_idx = int(np.random.choice(n, p=probs))

        next_center = P[next_idx]
        centers.append(next_center)

        # Update D[i] = min(D[i], dist(P[i], new_center)) — O(|P|)
        for i in range(n):
            d = sqdist(P[i], next_center)
            if d < D[i]:
                D[i] = d

    return centers


print("kmeansPP function defined.")

kmeansPP function defined.


In [ ]:
def kmeansObj(P, C):

    total = 0.0
    for p in P:
        # Find minimum squared distance to any center
        min_d = min(sqdist(p, c) for c in C)
        total += min_d
    return total / len(P)


print("kmeansObj function defined.")

kmeansObj function defined.


In [ ]:
k  = 5    # number of final centers
k1 = 25   # k1 > k; used as coreset size in experiment 3

random.seed(42)
np.random.seed(42)

print(f"Dataset size : |P| = {len(P)} points, each with {len(P[0])} dimensions")
print(f"Parameters   : k = {k}, k1 = {k1}")
print("=" * 60)
print("\n[Experiment 1] Running kcenter(P, k) ...")
t0 = time.time()
C1 = kcenter(P, k)
t1 = time.time()
print(f"  kcenter running time : {t1 - t0:.4f} seconds")
print(f"  Number of centers    : {len(C1)}")

# Also compute objective for reference
obj1 = kmeansObj(P, C1)
print(f"  kmeansObj(P, C1)     : {obj1:.6f}")
print("\n[Experiment 2] Running kmeansPP(P, k) ...")
t0 = time.time()
C2 = kmeansPP(P, k)
t_kmpp = time.time() - t0
print(f"  kmeansPP running time: {t_kmpp:.4f} seconds")

obj2 = kmeansObj(P, C2)
print(f"  kmeansObj(P, C2)     : {obj2:.6f}")
print("\n[Experiment 3] Coreset approach: kcenter(P, k1) → kmeansPP(X, k) → kmeansObj ...")
print(f"  Step 3a: Running kcenter(P, k1={k1}) to build coreset X ...")
t0 = time.time()
X = kcenter(P, k1)          # build a k1-center coreset
t_kc2 = time.time() - t0
print(f"  kcenter(P, k1) running time : {t_kc2:.4f} seconds, |X| = {len(X)}")

print(f"  Step 3b: Running kmeansPP(X, k={k}) on the coreset ...")
C3 = kmeansPP(X, k)         # run k-means++ on the small coreset

obj3 = kmeansObj(P, C3)     # evaluate on full P
print(f"  kmeansObj(P, C3)            : {obj3:.6f}")
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Exp 1 — kcenter(P, k={k})                        obj = {obj1:.6f}")
print(f"Exp 2 — kmeansPP(P, k={k})                       obj = {obj2:.6f}")
print(f"Exp 3 — kcenter(P,k1={k1}) → kmeansPP(X,k={k})  obj = {obj3:.6f}")
print("\nInterpretation:")
print("  A lower kmeansObj value indicates better cluster quality.")
print("  Experiment 3 tests if a larger kcenter coreset helps kmeans++ find better centers.")

Dataset size : |P| = 4601 points, each with 57 dimensions
Parameters   : k = 5, k1 = 25

[Experiment 1] Running kcenter(P, k) ...
  kcenter running time : 0.0542 seconds
  Number of centers    : 5
  kmeansObj(P, C1)     : 206586.640926

[Experiment 2] Running kmeansPP(P, k) ...
  kmeansPP running time: 0.0501 seconds
  kmeansObj(P, C2)     : 102200.090406

[Experiment 3] Coreset approach: kcenter(P, k1) → kmeansPP(X, k) → kmeansObj ...
  Step 3a: Running kcenter(P, k1=25) to build coreset X ...
  kcenter(P, k1) running time : 0.2542 seconds, |X| = 25
  Step 3b: Running kmeansPP(X, k=5) on the coreset ...
  kmeansObj(P, C3)            : 133364.697840

SUMMARY
Exp 1 — kcenter(P, k=5)                        obj = 206586.640926
Exp 2 — kmeansPP(P, k=5)                       obj = 102200.090406
Exp 3 — kcenter(P,k1=25) → kmeansPP(X,k=5)  obj = 133364.697840

Interpretation:
  A lower kmeansObj value indicates better cluster quality.
  Experiment 3 tests if a larger kcenter coreset helps kme

In [ ]:

def kmeansObj_spark(sc, P, C):
    # Broadcast centers so every worker node has a local copy
    bc_C = sc.broadcast(C)

    # Parallelise data points into an RDD
    rdd = sc.parallelize(P)

    # For each point compute its min squared distance to any center
    def min_sq_dist(p):
        centers = bc_C.value
        return min(float(np.dot(p - c, p - c)) for c in centers)

    total   = rdd.map(min_sq_dist).sum()
    n       = len(P)
    return total / n


print("Running Spark RDD kmeansObj for Experiment 2 centers ...")
t0 = time.time()
obj2_spark = kmeansObj_spark(sc, P, C2)
print(f"  Spark kmeansObj(P, C2) = {obj2_spark:.6f}  (took {time.time()-t0:.4f}s)")
print(f"  Sequential result      = {obj2:.6f}")
print(f"  Results match          : {abs(obj2_spark - obj2) < 1e-6}")

Running Spark RDD kmeansObj for Experiment 2 centers ...
  Spark kmeansObj(P, C2) = 102200.090406  (took 4.2319s)
  Sequential result      = 102200.090406
  Results match          : True
